In [ ]:
# --- Environment Setup (do not edit) ---
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print("\u26a0\ufe0f  WARNING: Could not detect platform. Falling back to .env (local).")
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"\u26a0\ufe0f  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"\u26a0\ufe0f  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"\u2705 Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"\u2705 Env file : {_env_file}")
print(f"\u2705 DATA_ROOT: {DATA_ROOT}")
print(f"{'\u2705' if DATA_ROOT.exists() else '\u274c'} Path exists: {DATA_ROOT.exists()}")
# ---------------------------------------------------------------------------

# ViFactCheck LLM Fine-Tuning (Stage 3.9c)

Fine-tunes **Gemma-7B** (paper's best model, 89.90% macro F1) on ViFactCheck using **Unsloth + LoRA**,
following the original `llm_training.py` from [TTHHA/ViFactCheck](https://github.com/TTHHA/ViFactCheck).

Paper §4.1: LLMs (Llama, Gemma, Mistral) are fine-tuned with LoRA (PEFT) on the instruction-formatted dataset.

- **Model**: `unsloth/gemma-7b-bnb-4bit` (4-bit quantized)
- **PEFT**: LoRA r=16, target all attention + MLP projections, lora_alpha=16, lora_dropout=0
- **Instruction format**: Alpaca prompt template (Instruction + Input + Response)
- **Dataset**: `tranthaihoa/data_instruct` (instruction-formatted ViFactCheck)
- **Training**: SFTTrainer, 5 epochs, lr=2e-4, batch_size=4, grad_accum=4
- **Hardware**: Requires CUDA GPU with >= 16GB VRAM (4-bit quantization)

```
Output: training/checkpoints_vifactcheck_llm/<run>/ (LoRA adapter + tokenizer)
```

> **Note**: Unsloth requires a CUDA GPU. This notebook will not work on MPS/CPU.
> Run on Colab (T4/A100) or Vast.ai.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck_llm",
    },
    "model": {
        "model_name": "unsloth/gemma-7b-bnb-4bit",
        "max_seq_length": 2048,
        "dtype": None,
        "load_in_4bit": True,
    },
    "peft": {
        "r": 16,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        "lora_alpha": 16,
        "lora_dropout": 0,
        "bias": "none",
        "use_gradient_checkpointing": "unsloth",
        "random_state": 3407,
        "use_rslora": False,
    },
    "data": {
        "hf_dataset": "tranthaihoa/data_instruct",
        "split": "train",
    },
    "training": {
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "warmup_steps": 5,
        "num_train_epochs": 5,
        "learning_rate": 2e-4,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 3407,
        "logging_steps": 1,
        "packing": False,
        "dataset_num_proc": 2,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Checkpoint:   {CONFIG['paths']['checkpoint_root']}")
print(f"Model:        {CONFIG['model']['model_name']}")
print(f"Dataset:      {CONFIG['data']['hf_dataset']}")

In [ ]:
# --- GPU CHECK ---
# Unsloth requires CUDA. This notebook cannot run on MPS/CPU.
import torch

if not torch.cuda.is_available():
    print("\u26a0\ufe0f  WARNING: No CUDA GPU detected. Unsloth requires CUDA.")
    print("   This notebook must be run on:")
    print("   - Google Colab (T4/A100)")
    print("   - Vast.ai (RTX 3090/A100/H100)")
    print("   - Any machine with NVIDIA GPU >= 16GB VRAM")
    raise RuntimeError("CUDA GPU required for Unsloth LLM fine-tuning.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\u2705 CUDA GPU: {gpu_name} ({gpu_mem:.1f} GB)")

if gpu_mem < 14:
    print(f"\u26a0\ufe0f  WARNING: GPU memory ({gpu_mem:.1f} GB) may be insufficient for 7B model.")
    print("   Consider using a smaller model (gemma-2b) or a GPU with >= 16GB VRAM.")

In [ ]:
# --- INSTALL UNSLOTH (if needed) ---
try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    print("Unsloth already installed.")
except ImportError:
    print("Installing Unsloth...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "unsloth"])
    from unsloth import FastLanguageModel, is_bfloat16_supported
    print("Unsloth installed.")

bf16 = is_bfloat16_supported()
print(f"bfloat16 supported: {bf16}")

## Step 1: Load Model and Tokenizer

Loads the 4-bit quantized Gemma-7B model via Unsloth's `FastLanguageModel`.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = CONFIG["model"]["max_seq_length"]
dtype = CONFIG["model"]["dtype"]
load_in_4bit = CONFIG["model"]["load_in_4bit"]
model_name = CONFIG["model"]["model_name"]

print(f"Loading model: {model_name}")
print(f"  max_seq_length: {max_seq_length}")
print(f"  load_in_4bit: {load_in_4bit}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model and tokenizer loaded.")

## Step 2: Configure LoRA (PEFT)

Applies LoRA adapters to all attention and MLP projection layers.
Matches original `llm_training.py` PEFT config exactly.

In [ ]:
peft_cfg = CONFIG["peft"]

model = FastLanguageModel.get_peft_model(
    model,
    r=peft_cfg["r"],
    target_modules=peft_cfg["target_modules"],
    lora_alpha=peft_cfg["lora_alpha"],
    lora_dropout=peft_cfg["lora_dropout"],
    bias=peft_cfg["bias"],
    use_gradient_checkpointing=peft_cfg["use_gradient_checkpointing"],
    random_state=peft_cfg["random_state"],
    use_rslora=peft_cfg["use_rslora"],
    loftq_config=None,
)

print("LoRA adapters applied.")
print(f"  r={peft_cfg['r']}  alpha={peft_cfg['lora_alpha']}  dropout={peft_cfg['lora_dropout']}")
print(f"  target_modules: {peft_cfg['target_modules']}")

## Step 3: Load and Format Instruction Dataset

Uses the Alpaca prompt template (same as original `llm_training.py`):
```
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
```

Dataset: `tranthaihoa/data_instruct` (instruction-formatted ViFactCheck with fields: instructions, input, output)

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    instructions = examples["instructions"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, out in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, inp, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


hf_dataset = CONFIG["data"]["hf_dataset"]
split = CONFIG["data"]["split"]

print(f"Loading dataset: {hf_dataset} (split={split})")
dataset = load_dataset(hf_dataset, split=split)
print(f"  Columns: {dataset.column_names}")
print(f"  Examples: {len(dataset)}")
print(f"  First sample: { {k: str(v)[:80] for k, v in dataset[0].items()} }")

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\nFormatted dataset: {len(dataset)} examples")
print(f"  First text (truncated): {dataset[0]['text'][:200]}...")

## Step 4: Configure SFTTrainer and Train

Uses `SFTTrainer` from `trl` with the same hyperparameters as the original `llm_training.py`:
- `per_device_train_batch_size=4`, `gradient_accumulation_steps=4`
- `num_train_epochs=5`, `learning_rate=2e-4`
- `optim="adamw_8bit"`, `lr_scheduler_type="linear"`
- `warmup_steps=5`, `weight_decay=0.01`

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"vifactcheck_llm_{timestamp}"
run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

train_cfg = CONFIG["training"]
bf16 = is_bfloat16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=train_cfg["dataset_num_proc"],
    packing=train_cfg["packing"],
    args=TrainingArguments(
        per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
        warmup_steps=train_cfg["warmup_steps"],
        num_train_epochs=train_cfg["num_train_epochs"],
        learning_rate=train_cfg["learning_rate"],
        fp16=not bf16,
        bf16=bf16,
        logging_steps=train_cfg["logging_steps"],
        optim=train_cfg["optim"],
        weight_decay=train_cfg["weight_decay"],
        lr_scheduler_type=train_cfg["lr_scheduler_type"],
        seed=train_cfg["seed"],
        output_dir=str(run_dir),
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")

## Step 5: Save LoRA Adapter and Tokenizer

In [ ]:
import json

# Save LoRA adapter
adapter_dir = run_dir / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"LoRA adapter saved: {adapter_dir}")

# Save merged model (optional -- requires more disk space)
# merged_dir = run_dir / "merged_model"
# model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
# print(f"Merged model saved: {merged_dir}")

# Save manifest
manifest = {
    "lora_adapter_path": str(adapter_dir),
    "model_name": CONFIG["model"]["model_name"],
    "max_seq_length": max_seq_length,
    "load_in_4bit": load_in_4bit,
    "peft_config": CONFIG["peft"],
    "training_setup": CONFIG["training"],
    "dataset": CONFIG["data"]["hf_dataset"],
    "training_loss": trainer_stats.training_loss,
    "total_steps": trainer_stats.global_step,
    "note": "Follows original ViFactCheck llm_training.py (AAAI 2025). Gemma-7B 4-bit LoRA.",
}
manifest_path = run_dir / "checkpoint_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written: {manifest_path}")
print(f"\nDone. All outputs in: {run_dir}")

## Step 6: Inference Test (Optional)

Quick test to verify the fine-tuned model can classify a sample claim.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# Test with a sample from the dataset
sample = dataset[0]
test_text = sample["text"]
# Truncate to just the prompt (before Response)
prompt_end = test_text.find("### Response:")
prompt_only = test_text[:prompt_end + len("### Response:")]

inputs = tokenizer(prompt_only, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("=== Inference Test ===")
print(f"Prompt (truncated): {prompt_only[:300]}...")
print(f"\nGenerated response: {response[len(prompt_only):].strip()}")
print(f"\nOriginal label: {sample.get('output', 'N/A')}")